# 1D J1-J2 Inference : PoincareGRU (seed 111-555) 

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('../../1_hypnqs_torch_poincare/utility_poincare')
from j1j2_hyprnn_train_loop import *
numsamples = 10000

GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test(wf,  numsamples,path_to_weights, Ee, clipped_e=False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [4]:
N=100
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname =f'res_PoincareGRU'

## J2=0.0

In [7]:
J2_ = 0.0
J2 = +J2_ * np.ones(syssize)
Ee_00=-44.12773986967251
rmax=1.0

In [9]:
seed=111
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'J2={J2_}_rmax={rmax}/seed={seed}/N100_J1=1.0|J2=0.0_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.07320022583008+0.008200000040233135j), var E = 1.9944000244140625
DMRG energy  is -44.1277
Time taken=0.324 hrs


In [6]:
seed=222
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'rmax=1.0/{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2=0.0_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 194 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.84560012817383-0.0071000000461936j), var E = 2.3817999362945557
DMRG energy  is -44.1277
Time taken=0.53 hrs


In [7]:
seed=333
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'rmax=1.0/{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2=0.0_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 316 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.071800231933594+0.0031999999191612005j), var E = 0.7900999784469604
DMRG energy  is -44.1277
Time taken=0.687 hrs


In [19]:
seed=444
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2=0.0_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_00)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.89820098876953-0.006500000134110451j), var E = 0.6234999895095825
DMRG energy  is -44.1277
Time taken=0.235 hrs


In [9]:
seed=555
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'rmax=1.0/{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2=0.0_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 560 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.96580123901367-0.000699999975040555j), var E = 1.4930000305175781
DMRG energy  is -44.1277
Time taken=0.481 hrs


## J2=0.2

In [11]:
J2_ = 0.2
J2 = +J2_ * np.ones(syssize)
Ee_02=-40.7388
rmax=1.0

In [12]:
seed=111
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'J2={J2_}_rmax={rmax}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.41899871826172+0.0005000000237487257j), var E = 0.2768999934196472
DMRG energy  is -40.7388
Time taken=0.585 hrs


In [5]:
seed=222
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 604 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.26020050048828-0.0038999998942017555j), var E = 0.2750999927520752
DMRG energy  is -40.7388
Time taken=0.527 hrs


In [6]:
seed=333
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 237 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.10200119018555+0.0010000000474974513j), var E = 0.1988999992609024
DMRG energy  is -40.7388
Time taken=0.507 hrs


In [7]:
seed=444
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 477 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.6265983581543+0.008299999870359898j), var E = 1.094099998474121
DMRG energy  is -40.7388
Time taken=0.559 hrs


In [8]:
seed=555
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 527 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.76890182495117-0.007300000172108412j), var E = 0.4343000054359436
DMRG energy  is -40.7388
Time taken=0.55 hrs


## J2=0.5

In [14]:
J2_ = 0.5
J2 = +J2_ * np.ones(syssize)
Ee_05=37.5000
rmax=1.0

In [18]:
seed=111
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'J2={J2_}_rmax={rmax}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 241 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.48609924316406+0.0007999999797903001j), var E = 0.030799999833106995
DMRG energy  is 37.5
Time taken=0.504 hrs


In [12]:
seed=222
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 276 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.48149871826172-0.001500000013038516j), var E = 0.027400000020861626
DMRG energy  is 37.5
Time taken=0.33 hrs


In [13]:
seed=333
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 167 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.42300033569336-0.0012000000569969416j), var E = 0.15459999442100525
DMRG energy  is 37.5
Time taken=0.329 hrs


In [14]:
seed=444
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 485 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.36389923095703+0.001500000013038516j), var E = 0.21529999375343323
DMRG energy  is 37.5
Time taken=0.329 hrs


In [10]:
seed=555
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=1.0,seed=seed)
h_wl = f'{fname}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax=1.0_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 559 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.593101501464844-0.0015999999595806003j), var E = 0.41339999437332153
DMRG energy  is 37.5
Time taken=0.572 hrs


## J2=0.8 (rmax=0.8)

In [19]:
J2_ = 0.8
J2 = +J2_ * np.ones(syssize)
Ee_08=-42.07006297371643
rmax=0.8

In [6]:
seed=111
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=rmax,seed=seed)
h_wl = f'rmax={rmax}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 649 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.428199768066406-0.008299999870359898j), var E = 0.49459999799728394
DMRG energy  is -42.0701
Time taken=0.721 hrs


In [7]:
seed=222
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=rmax,seed=seed)
h_wl = f'rmax={rmax}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_08)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.7041015625-0.00559999980032444j), var E = 1.8769999742507935
DMRG energy  is -42.0701
Time taken=0.718 hrs


In [11]:
seed=333
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=rmax,seed=seed)
h_wl = f'rmax={rmax}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 573 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.25669860839844+0.01080000028014183j), var E = 2.7564001083374023
DMRG energy  is -42.0701
Time taken=0.723 hrs


In [9]:
seed=444
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=rmax,seed=seed)
h_wl = f'rmax={rmax}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 448 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.677101135253906-0.0020000000949949026j), var E = 0.19120000302791595
DMRG energy  is -42.0701
Time taken=0.722 hrs


In [10]:
seed=555
set_cpu_deterministic(seed)
hGRU_00 = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypGRU', bias_geom='hyp',
                           hyp_non_lin='id', units=units, r_max=rmax,seed=seed)
h_wl = f'rmax={rmax}/J2={J2_}/seed={seed}/N100_J1=1.0|J2={J2_}_HypGRU_70_id_hyp_rmax={rmax}_ns=80_MsTrue_checkpoint.pt'
t0 = time.time()
define_load_test(hGRU_00, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Successfully remapped and loaded weights.
Clipped 511 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.40850067138672-0.004100000020116568j), var E = 0.7335000038146973
DMRG energy  is -42.0701
Time taken=0.715 hrs
